# Linear buckling analysis

Linear buckling loads of two laminated cylinders under axial compression,
the cylinders Z18 and Z33 of Castro et al. [Castro2014], Table 4. The
geometric stiffness matrix $\boldsymbol{K}_G$ is computed in two ways:

- from the membrane resultants of a linear static solution, with
  `update_KG`: the prebuckling state then contains the bending boundary
  layers at the supports and the hoop resultants of the restrained Poisson
  expansion;
- from a uniform membrane force given directly, with
  `update_KG_constant_stress`, without a static analysis.

The same analyses are checked by the tests
`tests/test_linear_buckling.py` (Z18),
`tests/test_linear_buckling_Z33_displ.py` and
`tests/test_linear_buckling_Z33_constant_stress.py` (Z33).

**Theory.** $\boldsymbol{K}_G$ is linear in the prebuckling displacements
$\boldsymbol{u}_0$, so the linear buckling problem, see
[Analyses](../analyses.rst),

$$
\left( \boldsymbol{K}_{C0} + \lambda \boldsymbol{K}_G(\boldsymbol{u}_0)
\right) \boldsymbol{\phi} = \boldsymbol{0}
$$

gives the multiplier $\lambda$ of the load that produced
$\boldsymbol{u}_0$, whatever its size. It is solved for the lowest
multipliers with the shift-invert mode `'buckling'` of `eigsh`. When the
load is a prescribed shortening of the end $x = L$, the static solution is
that of the unknown degrees of freedom with the known values
$\bar{\boldsymbol{u}}_k$,

$$
\boldsymbol{K}_{uu} \boldsymbol{u}_u = -\boldsymbol{K}_{uk}
\bar{\boldsymbol{u}}_k, \qquad
\boldsymbol{F}_k = \boldsymbol{K}_{uk}^\top \boldsymbol{u}_u +
\boldsymbol{K}_{kk} \bar{\boldsymbol{u}}_k
$$

and the critical load is $\lambda$ times the sum of the axial reactions
$\boldsymbol{F}_k$ at $x = L$.

In [1]:
import numpy as np
from numpy import isclose, pi
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh, spsolve
from composites import laminated_plate

from bfsccylinder import (BFSCCylinder, update_KC0, update_KG,
                          update_KG_constant_stress, DOF, DOUBLE, INT,
                          KC0_SPARSE_SIZE, KG_SPARSE_SIZE)
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Cylinders and material

Both cylinders have $R = 250$ mm, $L = 510$ mm and ten plies of 0.125 mm,
the stacking sequences of [Castro2014], Table 4, listed from the inner to
the outer surface. The ply material is Geier1997 of [Castro2014], Table 3,
the material of the test cylinders of Geier and Singh [GeierSingh1997] and
of Z33 in [Geier2002]. The element uses only the laminate stiffnesses
$\boldsymbol{A}$, $\boldsymbol{B}$, $\boldsymbol{D}$ of the classical
laminated plate theory; the two transverse shear moduli that
`laminated_plate` takes are not used and are given the value of $G_{12}$.

In [2]:
L = 0.510  # m
R = 0.250  # m
b = 2*pi*R  # m
E11, E22, nu12, G12 = 123.55e9, 8.708e9, 0.319, 5.695e9
laminaprop = (E11, E22, nu12, G12, G12, G12)
plyt = 0.125e-3  # m
stacks = {
    'Z18': [+60, -60, 0, 0, +68, -68, +52, -52, +37, -37][::-1],
    'Z33': [0, 0, 19, -19, 37, -37, 45, -45, 51, -51],
}
props = {k: laminated_plate(stack=s, plyt=plyt, laminaprop=laminaprop)
         for k, s in stacks.items()}

## Mesh, elements and matrices

The model is built by one function, used for both cylinders: the
structured mesh closed around the circumference (see
[Assembling a model](../assembling.rst)), one `BFSCCylinder` per element,
and the stiffness matrix $\boldsymbol{K}_{C0}$. It returns what the
analyses below need.

In [3]:
nint = 4
points, weights = get_points_weights(nint=nint)

def build_model(prop, ny):
    nx = int(ny*L/b)
    nids = 1 + np.arange(nx*(ny+1))
    nids_mesh = nids.reshape(nx, ny+1)
    nids_mesh[:, -1] = nids_mesh[:, 0]  # closes the cylinder
    nids = np.unique(nids_mesh)
    nid_pos = dict(zip(nids, np.arange(len(nids))))
    xmesh, ymesh = np.meshgrid(np.linspace(0, L, nx),
                               np.linspace(0, b, ny+1)[:-1], indexing='ij')
    n1s = nids_mesh[:-1, :-1].flatten()
    n2s = nids_mesh[1:, :-1].flatten()
    n3s = nids_mesh[1:, 1:].flatten()
    n4s = nids_mesh[:-1, 1:].flatten()
    N = DOF*nx*ny
    elements = []
    for k, (n1, n2, n3, n4) in enumerate(zip(n1s, n2s, n3s, n4s)):
        shell = BFSCCylinder(nint)
        shell.c1 = DOF*nid_pos[n1]
        shell.c2 = DOF*nid_pos[n2]
        shell.c3 = DOF*nid_pos[n3]
        shell.c4 = DOF*nid_pos[n4]
        shell.R = R
        shell.lex = L/(nx - 1)
        shell.ley = b/ny
        assign_constant_ABD(shell, prop)
        shell.init_k_KC0 = k*KC0_SPARSE_SIZE
        shell.init_k_KG = k*KG_SPARSE_SIZE
        elements.append(shell)
    ne = len(elements)
    Kr = np.zeros(KC0_SPARSE_SIZE*ne, dtype=INT)
    Kc = np.zeros(KC0_SPARSE_SIZE*ne, dtype=INT)
    Kv = np.zeros(KC0_SPARSE_SIZE*ne, dtype=DOUBLE)
    for shell in elements:
        update_KC0(shell, points, weights, Kr, Kc, Kv)
    KC0 = coo_matrix((Kv, (Kr, Kc)), shape=(N, N)).tocsc()
    return dict(nx=nx, ny=ny, N=N, x=xmesh.flatten(), y=ymesh.flatten(),
                xmesh=xmesh, ymesh=ymesh, elements=elements, KC0=KC0)

def new_KG(model):
    size = KG_SPARSE_SIZE*len(model['elements'])
    return (np.zeros(size, dtype=INT), np.zeros(size, dtype=INT),
            np.zeros(size, dtype=DOUBLE))

def constrain_ends(model, dofs):
    bk = np.zeros(model['N'], dtype=bool)
    ends = isclose(model['x'], 0) | isclose(model['x'], L)
    for dof in dofs:
        bk[dof::DOF] = ends
    return bk

The two analyses, as functions of the model and of the constrained degrees
of freedom at both ends:

In [4]:
def buckling_shortening(model, dofs, shortening=-0.001):
    """K_G of the linear static solution of a prescribed end shortening"""
    N, x, KC0 = model['N'], model['x'], model['KC0']
    bk = constrain_ends(model, dofs)
    bu = ~bk
    top = isclose(x, L)
    u = np.zeros(N)
    u[0::DOF][top] = shortening
    uk = u[bk]
    Kuk = KC0[bu][:, bk]
    u[bu] = spsolve(KC0[bu][:, bu], -Kuk @ uk)
    KGr, KGc, KGv = new_KG(model)
    for shell in model['elements']:
        update_KG(u, shell, points, weights, KGr, KGc, KGv)
    KG = coo_matrix((KGv, (KGr, KGc)), shape=(N, N)).tocsc()
    eigvals, eigvecs = eigsh(A=KC0[bu][:, bu], k=3, which='SM',
                             M=KG[bu][:, bu], sigma=1., mode='buckling')
    # reactions at the known degrees of freedom
    f = np.zeros(N)
    f[bk] = Kuk.T @ u[bu] + KC0[bk][:, bk] @ uk
    Pcr = -eigvals[0]*f[0::DOF][top].sum()
    mode = np.zeros(N)
    mode[bu] = eigvecs[:, 0]
    return Pcr, mode

def buckling_constant_stress(model, dofs, Nxx=-1.):
    """K_G of a uniform axial membrane force Nxx"""
    N, KC0 = model['N'], model['KC0']
    bk = constrain_ends(model, dofs)
    bu = ~bk
    KGr, KGc, KGv = new_KG(model)
    for shell in model['elements']:
        update_KG_constant_stress(shell, points, weights, KGr, KGc, KGv,
                                  Nxx, 0., 0.)
    KG = coo_matrix((KGv, (KGr, KGc)), shape=(N, N)).tocsc()
    eigvals, eigvecs = eigsh(A=KC0[bu][:, bu], k=3, which='SM',
                             M=KG[bu][:, bu], sigma=1., mode='buckling')
    Pcr = -eigvals[0]*Nxx*b
    mode = np.zeros(N)
    mode[bu] = eigvecs[:, 0]
    return Pcr, mode

def plot_mode(model, mode, title):
    w = mode[6::DOF].reshape(model['nx'], model['ny'])
    fig, ax = plt.subplots(figsize=(7, 3.0))
    c = ax.contourf(model['ymesh']*1e3, model['xmesh']*1e3, w/np.abs(w).max(),
                    levels=50, cmap='coolwarm')
    fig.colorbar(c, ax=ax, label='$w / |w|_{max}$')
    ax.set_xlabel('$y$ (mm), around the circumference')
    ax.set_ylabel('$x$ (mm)')
    ax.set_aspect('equal')
    ax.set_title(title)
    plt.show()

## Cylinder Z18, simply supported


The ends have $u = v = w = 0$ at the nodes and $w_{,x}$ free, the
conditions SS1 of [Castro2014], Table 1; between two nodes of an end $w$ is
the cubic of the nodal $w$ and $w_{,y}$, so $w = 0$ holds exactly only at
the nodes, a difference that vanishes with mesh refinement, see
[Assembling a model](../assembling.rst).

The end $x = L$ is shortened by 1 mm; 60 nodes around the circumference.

In [5]:
z18 = build_model(props['Z18'], ny=60)
P_z18, mode_z18 = buckling_shortening(z18, dofs=(0, 3, 6))
print(f'Z18: P_cr = {abs(P_z18)/1e3:.2f} kN')
plot_mode(z18, mode_z18, f'Z18, $P_{{cr}}$ = {abs(P_z18)/1e3:.1f} kN')

Z18: P_cr = 229.06 kN


## Cylinder Z33, clamped

The ends are clamped, $u = v = w = w_{,x} = 0$, the conditions CC1 of
[Castro2014], Table 1, with 50 nodes around the circumference. Unlike the
simple support of Z18, the derivatives along the edge, $u_{,y}$, $v_{,y}$,
$w_{,y}$ and $w_{,xy}$, are constrained too, so that the conditions hold
along the whole edge: at a clamped end the deviation between the nodes is
larger, see [Assembling a model](../assembling.rst). The
static prebuckling state of a prescribed shortening and the uniform
membrane force are compared.

In [6]:
z33 = build_model(props['Z33'], ny=50)
clamped = (0, 2, 3, 5, 6, 7, 8, 9)  # u, u_y, v, v_y, w, w_x, w_y, w_xy
P_z33_s, mode_z33 = buckling_shortening(z33, dofs=clamped)
P_z33_c, _ = buckling_constant_stress(z33, dofs=clamped)
print(f'Z33, static prebuckling state: P_cr = {abs(P_z33_s)/1e3:.2f} kN')
print(f'Z33, uniform membrane force:   P_cr = {abs(P_z33_c)/1e3:.2f} kN')
plot_mode(z33, mode_z33, f'Z33, $P_{{cr}}$ = {abs(P_z33_s)/1e3:.1f} kN')

Z33, static prebuckling state: P_cr = 201.69 kN
Z33, uniform membrane force:   P_cr = 201.75 kN


## Comparison

- Z33: the buckling load of Geier et al. [Geier2002], Table 1, 192 kN, the
  result BACCUS-C4 (built-in edges, deep shell theory, homogeneous membrane
  prebuckling state); the tests of the same paper gave 173 kN and 184 kN.
- Z18: [Castro2014], Table 6, gives with the classical laminated plate
  theory 226.44 kN for the conditions SS2 ($v = w = 0$, $u$ free) and
  228.22 kN for CC2 ($v = w = w_{,x} = 0$), and the test result 212.6 kN of
  Geier et al.; the conditions here, with $u$ also constrained, are between
  the two.

The table also gives the regression values stored in the tests, results of
the same models.

In [7]:
rows = [
    ('Z18', 'static', P_z18, 226.44e3, 'Castro et al. (2014), Table 6, SS2'),
    ('Z18', 'static', P_z18, 228.22e3, 'Castro et al. (2014), Table 6, CC2'),
    ('Z18', 'static', P_z18, 228459.04645094523, 'regression value of the test'),
    ('Z33', 'static', P_z33_s, 192e3, 'Geier et al. (2002), Table 1'),
    ('Z33', 'static', P_z33_s, 201093., 'result stated in the Notes of the test'),
    ('Z33', 'membrane', P_z33_c, 192e3, 'Geier et al. (2002), Table 1'),
    ('Z33', 'membrane', P_z33_c, 201751., 'regression value of the test'),
]
table([{'cylinder': c, 'prebuckling': p, 'P_cr (kN)': abs(P)/1e3,
               'reference (kN)': ref/1e3, 'source': s,
               'rel. diff. (%)': 100*(abs(P) - ref)/ref}
              for c, p, P, ref, s in rows])

| cylinder | prebuckling | P_cr (kN) | reference (kN) | source | rel. diff. (%) |
|---|---|---|---|---|---|
| Z18 | static | 229.06 | 226.44 | Castro et al. (2014), Table 6, SS2 | 1.1587 |
| Z18 | static | 229.06 | 228.22 | Castro et al. (2014), Table 6, CC2 | 0.36976 |
| Z18 | static | 229.06 | 228.46 | regression value of the test | 0.26474 |
| Z33 | static | 201.69 | 192 | Geier et al. (2002), Table 1 | 5.0454 |
| Z33 | static | 201.69 | 201.09 | result stated in the Notes of the test | 0.2955 |
| Z33 | membrane | 201.75 | 192 | Geier et al. (2002), Table 1 | 5.0787 |
| Z33 | membrane | 201.75 | 201.75 | regression value of the test | 2.5359e-05 |

## References

- **[Castro2014]** Castro, S. G. P., Mittelstedt, C., Monteiro, F. A. C., Arbelo, M. A.,
  Ziegmann, G., and Degenhardt, R., Linear buckling predictions of unstiffened
  laminated composite cylinders and cones under various loading and boundary
  conditions using semi-analytical models, Composite Structures, 118, 303-315,
  2014, doi:10.1016/j.compstruct.2014.07.037.
- **[Geier2002]** Geier, B., Meyer-Piening, H.-R., and Zimmermann, R., On the influence of
  laminate stacking on buckling of composite cylindrical shells subjected to
  axial compression, Composite Structures, 55(4), 467-474, 2002,
  doi:10.1016/S0263-8223(01)00175-1.
- **[GeierSingh1997]** Geier, B., and Singh, G., Some simple solutions for buckling loads of thin
  and moderately thick cylindrical shells and panels made of laminated
  composite material, Aerospace Science and Technology, 1(1), 47-63, 1997,
  doi:10.1016/S1270-9638(97)90023-7.